# Bài học 11: Nối chuỗi Agent và Pipeline

## Agent Chaining là gì?

**Agent chaining** là khi đầu ra của **agent này** trở thành đầu vào cho **agent tiếp theo**.

Hiểu đơn giản, đó là **dây chuyền lắp ráp** trong nhà máy:
- Công nhân 1 làm bước 1, chuyển sang công nhân 2
- Công nhân 2 làm bước 2, chuyển sang công nhân 3
- ...

Trong AI:
- Researcher Agent **tìm kiếm thông tin** rồi chuyển cho Writer Agent
- Writer Agent **viết bài** từ thông tin đó rồi chuyển cho Editor Agent
- ...

Mỗi agent chỉ **làm tốt một việc duy nhất**, rồi bàn giao cho agent kế tiếp.

> **Chi phí:** ~$0.05-0.10 (2 lần gọi Sonnet nối tiếp). Mất 30-60 giây. Chạy mỗi ô một lần.

## Kế hoạch: 2 Agent nối chuỗi

Chúng ta sẽ xây dựng:

```
[Agent 1: Researcher]  →  ghi chú nghiên cứu  →  [Agent 2: Writer]
    Tìm kiếm trên web                              Viết bài từ nghiên cứu
```

- **Agent 1 (Researcher):** Có tool DataForSEO search (cùng tool từ Bài 9)
- **Agent 2 (Writer):** Nhận kết quả nghiên cứu, viết thành đoạn văn

Cách nối chuỗi rất đơn giản:
```python
research = researcher.run("Search for info...")
article = writer.run(f"Write from: {research.content}")  # Truyền đầu ra làm đầu vào!
```

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../../output/backend"))

from dotenv import load_dotenv
load_dotenv()

from agno.agent import Agent
from agno.models.anthropic import Claude
from tools.search import DataForSEOSearchTools
from tools.aio import get_dataforseo_credentials

# Thiết lập search tools (giống Bài 9)
creds = get_dataforseo_credentials()
search_tools = []
if creds:
    search_tools = [DataForSEOSearchTools(login=creds[0], password=creds[1])]
    print("DataForSEO search đã bật!")
else:
    print("Không có DataForSEO key — researcher sẽ dùng kiến thức có sẵn.")

# Agent 1: Researcher — tìm kiếm trên web
researcher = Agent(
    name="Researcher",
    model=Claude(id="claude-sonnet-4-5-20250929"),
    tools=search_tools,
    instructions=["Search the web and return detailed research notes."],
)

# Agent 2: Writer — viết từ nghiên cứu
writer = Agent(
    name="Writer",
    model=Claude(id="claude-sonnet-4-5-20250929"),
    instructions=[
        "Write a short paragraph (3-4 sentences) based on the research notes provided.",
        "Write clearly and professionally.",
    ],
)

print("Đã tạo 2 agent: Researcher và Writer")

In [ ]:
# Bước 1: Nghiên cứu
print("Bước 1: Đang nghiên cứu...")
research = researcher.run("Find information about AI trends in SEO for 2026")
print(f"Nghiên cứu xong! ({len(research.content)} ký tự)\n")

# Bước 2: Viết bài từ nghiên cứu
print("Bước 2: Đang viết bài...")
article = writer.run(f"Write a paragraph from these research notes:\n\n{research.content}")
print(f"Bài viết:\n{article.content}")

## Điều Gì Vừa Xảy Ra?

Hãy theo dõi luồng xử lý:

1. **Researcher** nhận câu hỏi, tìm kiếm web qua DataForSEO, và trả về `research.content`
2. `research.content` được **truyền vào** prompt của Writer
3. **Writer** nhận kết quả nghiên cứu và viết thành đoạn văn hoàn chỉnh

**Dòng code then chốt:**
```python
article = writer.run(f"Write a paragraph from these research notes:\n\n{research.content}")
```

Chỉ cần truyền `research.content` (đầu ra của agent trước) vào `writer.run()` — vậy là xong!

## Sản phẩm thực tế so sánh thế nào

Dự án `agentic-content-seo` của chúng ta sử dụng biến thể của pattern này:

- **Content Writer** — Nghiên cứu VÀ viết trong cùng một agent (chain nội bộ: search → viết bài)
- **Image Finder** — Đọc bài viết, tìm hình ảnh, chèn vào, lưu bài viết đã cập nhật
- **AIO Analyzer** — Đọc bài viết và phân tích so với Google AI Overviews

Thay vì chain 4 agent riêng lẻ, sản phẩm kết hợp nghiên cứu và viết bài vào **một agent đủ năng lực** (Content Writer). Sau đó Image Finder và AIO Analyzer bổ sung cho bài viết.

**Nguyên lý chaining** vẫn áp dụng — Image Finder đọc kết quả của Content Writer, giống như Writer đọc kết quả của Researcher ở trên.

In [ ]:
# Luồng sản phẩm thực tế
print("""
Luồng sản phẩm thực tế:

  [Content Writer] ──bài viết──> [Image Finder] ──bài viết + hình──> [AIO Analyzer]
       │                              │                                    │
  Search + Viết bài          Tìm & chèn hình ảnh             Phân tích AI Overviews
""")

## Phần 2: Từ agent đơn giản đến pipeline thực tế

Chuỗi 2 agent ở trên rất đơn giản: text thuần vào, text thuần ra. Nhưng sản phẩm thực cần **dữ liệu có cấu trúc** chảy giữa các agent.

Ở Bài 10, bạn đã học rằng `output_schema` buộc agent trả về dữ liệu có cấu trúc. Khi kết hợp với chaining, bạn có một **pipeline** — các agent truyền dữ liệu có cấu trúc cho nhau.

### Schema lồng nhau (Nested Schemas)

Schema phẳng như `list[str]` phù hợp cho dàn ý đơn giản. Nhưng nếu mỗi phần cần tiêu đề, điểm chính, VÀ tóm tắt? Bạn cần **schema lồng nhau**:

```python
class Section(BaseModel):          # Một phần
    heading: str
    key_points: list[str]

class DetailedOutline(BaseModel):  # Dàn ý đầy đủ với các phần
    title: str
    sections: list[Section]        # DANH SÁCH các object Section
```

Đây là cùng khái niệm với dict bên trong list bên trong dict — nhưng Pydantic **tự động kiểm tra** cấu trúc cho bạn.

## Bài tập

Điền vào chỗ trống để thêm một **agent thứ ba** vào chuỗi: một Editor để cải thiện đầu ra của Writer.

1. Tạo agent `editor` với instructions kiểu: "You are a content editor. Improve the given text: fix grammar, make it more engaging, and ensure it's SEO-friendly. Return only the improved text."
2. Nối chuỗi: truyền `article.content` vào `editor.run()`
3. In ra cả phiên bản gốc lẫn phiên bản đã chỉnh sửa

Đây chính là pattern mà pipeline thực tế sử dụng — chỉ là với nhiều agent hơn!

In [ ]:
# Bài tập: Điền vào chỗ trống để thêm agent thứ ba vào chuỗi

from agno.agent import Agent
from agno.models.anthropic import Claude

# Agent editor — cải thiện đầu ra của writer
editor = Agent(
    name=___,                                    # Đặt tên cho agent
    model=Claude(id="claude-sonnet-4-5-20250929"),
    instructions=[___],                          # Hướng dẫn agent chỉnh sửa và cải thiện văn bản
)

# Nối chuỗi: truyền đầu ra của writer cho editor
# (giả sử biến `article` đã có từ các ô phía trên)
edited = editor.run(f"Improve this text:\n\n{___}")      # Truyền đầu ra của writer

print("=== BẢN GỐC ===")
print(article.content)
print("\n=== ĐÃ CHỈNH SỬA ===")
print(___)

<details>
<summary>Bấm để xem đáp án</summary>

```python
from agno.agent import Agent
from agno.models.anthropic import Claude

editor = Agent(
    name="Editor",
    model=Claude(id="claude-sonnet-4-5-20250929"),
    instructions=[
        "You are a content editor. Improve the given text: fix grammar, make it more engaging, and ensure it's SEO-friendly.",
        "Return only the improved text.",
    ],
)

edited = editor.run(f"Improve this text:\n\n{article.content}")

print("=== BẢN GỐC ===")
print(article.content)
print("\n=== ĐÃ CHỈNH SỬA ===")
print(edited.content)
```
</details>

---

## Tổng kết Bài 11

### Nối chuỗi Agent:
- **Agent chaining**: đầu ra của agent này trở thành đầu vào cho agent tiếp theo
- Cách thực hiện: truyền `response.content` vào `agent.run()`
- Mỗi agent chỉ cần **làm tốt một nhiệm vụ duy nhất**

### Schema lồng nhau cho Pipeline:
- **Pydantic model lồng nhau** cho phép truyền dữ liệu có cấu trúc giữa các agent
- `Section` bên trong `DetailedOutline` — mỗi phần có tiêu đề và điểm chính
- Sản phẩm thực kết hợp chaining + structured output để tạo pipeline đáng tin cậy

### Mô-đun 3 — Xây dựng Agent:

| Bài học | Khái niệm |
|---------|----------|
| Bài 8 | Tạo agent cơ bản với `name`, `model`, `instructions` |
| Bài 9 | Thêm **tools** để agent hành động (tìm kiếm web + khái niệm API) |
| Bài 10 | **Structured output** — buộc agent trả về dữ liệu có cấu trúc |
| Bài 11 | **Nối chuỗi agent** + **schema lồng nhau** — xây dựng pipeline |

**Bài tiếp theo:** Lưu trữ — cách bài viết được lưu cục bộ để kết quả làm việc của agent không bị mất.

---

## Sẵn sàng cho Mô-đun 4?

Sau bài tiếp theo (Lưu trữ), bạn sẽ chuyển sang Mô-đun 4 nơi bạn học **phát triển với sự hỗ trợ của AI** — sử dụng Claude Code để xây dựng sản phẩm thực.

Ba câu hỏi kiểm tra hiểu biết:
1. Nếu `response.content` của agent là text thuần, làm sao truyền nó cho agent tiếp theo?
2. Sự khác biệt giữa `list[str]` và `list[Section]` khi dùng làm schema là gì?
3. Tại sao sản phẩm thực sử dụng một Content Writer thay vì tách riêng Researcher + Writer?